In [ ]:
import os
from pathlib import Path
from tqdm.notebook import tqdm
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch



In [ ]:
# TOKENIZING DATA WITH SPECIAL BERT TOKENIZER

# ADD HF TOKEN HERE TO MAKE IT WORK

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

CHUNK_SIZE = 10000
INPUT_CSV = "995,000_rows.csv"
OUTPUT_DIR = Path("bert_chunks")
OUTPUT_DIR.mkdir(exist_ok=True)

for i, chunk in enumerate(tqdm(pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE), total=995000//CHUNK_SIZE)):
    
    texts = chunk["content"].fillna("").tolist()
    
    encoded = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None
    )
    
    result = pd.DataFrame({
        "input_ids":      encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "token_type_ids": encoded["token_type_ids"],
        "label":          chunk["type"].values,
    })
    
    table = pa.Table.from_pandas(result)
    pq.write_table(table, OUTPUT_DIR / f"chunk_{i:03d}.parquet")

print(f"Done! Written to {OUTPUT_DIR}")



In [ ]:
import random
from pathlib import Path

# Get all chunk files and shuffle
chunk_files = sorted(Path("bert_chunks").glob("*.parquet"))
random.seed(42)
random.shuffle(chunk_files)

# Split 80/10/10
n = len(chunk_files)
train_files = chunk_files[:int(n * 0.8)]
val_files   = chunk_files[int(n * 0.8):int(n * 0.9)]
test_files  = chunk_files[int(n * 0.9):]

print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

lbls = ['reliable', 'political', 'bias', 'fake', 'conspiracy', 'rumor', 'unknown', 'unreliable', 'clickbait', 'junksci', 'satire', 'hate']
label2id = {label: idx for idx, label in enumerate(lbls)}
id2label  = {idx: label for idx, label in enumerate(lbls)}

# Load splits
def load_parquet_chunks(files):
    dfs = []
    for f in tqdm(files):
        dfs.append(pd.read_parquet(f))
    data = pd.concat(dfs, ignore_index=True)
    del dfs
    data = data[data["label"].isin(lbls)]  # drop the date and anything else
    data["label"] = data["label"].map(label2id)
    return data

train_data = load_parquet_chunks(train_files)
val_data   = load_parquet_chunks(val_files)
test_data  = load_parquet_chunks(test_files)



print(train_data['label'].value_counts())


In [ ]:
class NewsDataset(Dataset):
    def __init__(self, data):
        self.input_ids      = data["input_ids"].tolist()
        self.attention_mask = data["attention_mask"].tolist()
        self.token_type_ids = data["token_type_ids"].tolist()
        self.labels         = data["label"].tolist()

    def __getitem__(self, idx):
        return {
            "input_ids":      torch.tensor(self.input_ids[idx]),
            "attention_mask": torch.tensor(self.attention_mask[idx]),
            "token_type_ids": torch.tensor(self.token_type_ids[idx]),
            "labels":         torch.tensor(self.labels[idx]),
        }

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_data)
val_dataset   = NewsDataset(val_data)
test_dataset  = NewsDataset(test_data)

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(lbls),
    id2label=id2label,
    label2id=label2id
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./bert_fakenews",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

# Evaluate on test set
predictions = trainer.predict(test_dataset)
y_pred = predictions.predictions.argmax(axis=1)
y_test = test_data["label"].tolist()
print(classification_report(y_test, y_pred, target_names=lbls))

model.save_pretrained("./bert_fakenews_final")
tokenizer.save_pretrained("./bert_fakenews_final")

In [ ]:
from transformers import BertModel
import torch
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# Load BERT without classification head
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()

def get_embeddings(data, batch_size=32):
    all_embeddings = []
    
    for i in tqdm(range(0, len(data), batch_size)):
        batch = data.iloc[i:i+batch_size]
        
        input_ids      = torch.tensor(batch["input_ids"].tolist())
        attention_mask = torch.tensor(batch["attention_mask"].tolist())
        token_type_ids = torch.tensor(batch["token_type_ids"].tolist())
        
        with torch.no_grad():
            outputs = bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        
        # Use [CLS] token embedding as sentence representation
        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_embeddings)
    
    return np.vstack(all_embeddings)

print("Getting train embeddings...")
X_train_emb = get_embeddings(train_data)
print("Getting val embeddings...")
X_val_emb   = get_embeddings(val_data)
print("Getting test embeddings...")
X_test_emb  = get_embeddings(test_data)

y_train = train_data["label"].tolist()
y_val   = val_data["label"].tolist()
y_test  = test_data["label"].tolist()

# Train SVM
svm = LinearSVC(max_iter=1000)
svm.fit(X_train_emb, y_train)

# Evaluate
y_pred = svm.predict(X_test_emb)
print(classification_report(y_test, y_pred, target_names=lbls))

In [ ]:
# 1. Install dependencies (run this in a Kaggle cell first)
!pip install transformers torch -q


import pandas as pd
import numpy as np
import torch
import random
from pathlib import Path
from tqdm import tqdm
from transformers import BertModel
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import pickle


# 3. Load parquet chunks — point to your Kaggle dataset path
CHUNKS_DIR = Path("/kaggle/input/datasets/lassehelmer/bert-fakenews/bert_chunks/")
chunk_files = sorted(CHUNKS_DIR.glob("*.parquet"))

random.seed(42)
random.shuffle(chunk_files)

n = len(chunk_files)
train_files = chunk_files[:int(n * 0.8)]
val_files   = chunk_files[int(n * 0.8):int(n * 0.9)]
test_files  = chunk_files[int(n * 0.9):]

lbls = ['reliable', 'political', 'bias', 'fake', 'conspiracy', 'rumor', 'unknown', 'unreliable', 'clickbait', 'junksci', 'satire', 'hate']
label2id = {label: idx for idx, label in enumerate(lbls)}

def load_parquet_chunks(files):
    dfs = []
    for f in tqdm(files):
        dfs.append(pd.read_parquet(f))
    data = pd.concat(dfs, ignore_index=True)
    del dfs
    data = data[data["label"].isin(lbls)]
    data["label"] = data["label"].map(label2id)
    return data

# 4. Get embeddings chunk by chunk to avoid OOM
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def get_embeddings_from_files(files, batch_size=64):
    all_embeddings = []
    all_labels = []
    
    for f in tqdm(files):
        df = pd.read_parquet(f)
        df = df[df["label"].isin(lbls)]
        df["label"] = df["label"].map(label2id)
        
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size]
            
            input_ids      = torch.tensor(batch["input_ids"].tolist()).to(device)
            attention_mask = torch.tensor(batch["attention_mask"].tolist()).to(device)
            token_type_ids = torch.tensor(batch["token_type_ids"].tolist()).to(device)
            
            with torch.no_grad():
                outputs = bert(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )
            
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_embeddings)
            all_labels.extend(batch["label"].tolist())
    
    return np.vstack(all_embeddings), np.array(all_labels)

print("Getting train embeddings...")
X_train_emb, y_train = get_embeddings_from_files(train_files)
print("Getting val embeddings...")
X_val_emb,   y_val   = get_embeddings_from_files(val_files)
print("Getting test embeddings...")
X_test_emb,  y_test  = get_embeddings_from_files(test_files)

# 5. Save embeddings so you don't have to recompute
np.save("/kaggle/working/X_train_emb.npy", X_train_emb)
np.save("/kaggle/working/X_val_emb.npy",   X_val_emb)
np.save("/kaggle/working/X_test_emb.npy",  X_test_emb)
np.save("/kaggle/working/y_train.npy", y_train)
np.save("/kaggle/working/y_val.npy",   y_val)
np.save("/kaggle/working/y_test.npy",  y_test)

# 6. Train SVM
svm = LinearSVC(max_iter=1000)
svm.fit(X_train_emb, y_train)

# Save the SVM model
with open("/kaggle/working/svm_model.pkl", "wb") as f:
    pickle.dump(svm, f)

# 7. Evaluate
y_pred = svm.predict(X_test_emb)
print(classification_report(y_test, y_pred, target_names=lbls))
